# PFDM 2次元Helmholtz方程式: 直接法 vs CG (スパース+BTTB+FFT)

2次元Helmholtz方程式 $-\\Delta u + k^2 u = f$ に対する
$(D_h K D_h^\\top + \\Sigma)\\alpha = b$ を解く：

- **直接法 (Cholesky)**: $D_h K D_h^\\top + \\Sigma$ を明示構築してCholesky $O(N^3)$
- **CG + スパース + 2D FFT** (提案手法):
  1. $w_1 = D_h^\\top v$ → スパース積 $O(N)$
  2. $w_2 = K w_1$ → BTTB → 2D FFT で $O(N \\log N)$
  3. $w_3 = D_h w_2 + \\Sigma v$ → スパース積 $O(N)$

$D_h = -\\Delta_h + k^2 I$, $N = N_x^2$


In [2]:
using LinearAlgebra
using SparseArrays
using FFTW
using SpecialFunctions
using IterativeSolvers
using LinearMaps
using Plots
using Statistics
using Printf


In [ ]:
# 2次元 Matérnカーネル（変更なし）
function matern_kernel_2d(x1, x2; ν=1.5, ℓ=1.0, σf=1.0)
    r = sqrt(sum((x1 .- x2).^2))
    r == 0 && return σf^2
    z = sqrt(2ν) * r / ℓ
    if ν == 0.5
        return σf^2 * exp(-z)
    elseif ν == 1.5
        return σf^2 * (1 + z) * exp(-z)
    elseif ν == 2.5
        return σf^2 * (1 + z + z^2/3) * exp(-z)
    else
        return σf^2 * (2^(1-ν)/gamma(ν)) * z^ν * besselk(ν, z)
    end
end

# 2次元 D_h = -Δ_h + k^2*I (Helmholtz, スパース)
# 対角: 4/dx^2 + k^2, オフ対角: -1/dx^2
function build_Dh_2d_helmholtz(Nx, Ny, dx, k)
    N = Nx * Ny
    index(i, j) = i + (j-1)*Nx
    rows, cols, vals = Int[], Int[], Float64[]
    for j in 1:Ny, i in 1:Nx
        ki = index(i, j)
        push!(rows, ki); push!(cols, ki); push!(vals, 4.0/dx^2 + k^2)
        if i > 1;  push!(rows, ki); push!(cols, index(i-1,j)); push!(vals, -1.0/dx^2); end
        if i < Nx; push!(rows, ki); push!(cols, index(i+1,j)); push!(vals, -1.0/dx^2); end
        if j > 1;  push!(rows, ki); push!(cols, index(i,j-1)); push!(vals, -1.0/dx^2); end
        if j < Ny; push!(rows, ki); push!(cols, index(i,j+1)); push!(vals, -1.0/dx^2); end
    end
    return sparse(rows, cols, vals, N, N)
end

# BTTB 行列ベクトル積・FFTセットアップ（変更なし）
function bttb_matvec(k_fft2, v, Nx, Ny)
    V     = reshape(v, Nx, Ny)
    V_pad = zeros(2Nx, 2Ny)
    V_pad[1:Nx, 1:Ny] .= V
    result = real.(ifft2(k_fft2 .* fft2(V_pad)))
    return vec(result[1:Nx, 1:Ny])
end

function bttb_fft_setup(k_col_mat, Nx, Ny)
    C = zeros(2Nx, 2Ny)
    C[1:Nx, 1:Ny] .= k_col_mat
    Nx > 1 && (C[Nx+2:2Nx, 1:Ny]   .= k_col_mat[Nx:-1:2, :])
    Ny > 1 && (C[1:Nx, Ny+2:2Ny]   .= k_col_mat[:, Ny:-1:2])
    Nx > 1 && Ny > 1 && (C[Nx+2:2Nx, Ny+2:2Ny] .= k_col_mat[Nx:-1:2, Ny:-1:2])
    return fft2(C)
end

fft2  = x -> fft(fft(x, 1), 2)
ifft2 = x -> ifft(ifft(x, 1), 2)

# 真の解・右辺: -Δu + k^2*u = f
# u(x,y) = sin(πx)sin(πy) → (-Δ + k^2)u = (2π^2 + k^2)u
u_exact(x, y, k)  = sin(π*x) * sin(π*y)
f_func(x, y, k)   = (2π^2 + k^2) * sin(π*x) * sin(π*y)


In [ ]:
ν      = 1.5
ℓ      = 1.0
σf     = 1.0
C      = 1e-3
τ      = 2
k_wave = 1.0

dense_max_N = 2500
grid_sizes = [5, 8, 10, 15, 20, 30, 50, 70, 100, 150, 200]

results = []

for Nx in grid_sizes
    Ny  = Nx
    N   = Nx * Ny
    dx  = 1.0 / (Nx + 1)

    x = range(dx, step=dx, length=Nx)
    y = range(dx, step=dx, length=Ny)
    X_all  = [[xi, yj] for yj in y for xi in x]
    b_vec  = [f_func(xi, yj, k_wave)  for yj in y for xi in x]
    u_true = [u_exact(xi, yj, k_wave) for yj in y for xi in x]

    Dh_sp   = build_Dh_2d_helmholtz(Nx, Ny, dx, k_wave)
    σ_noise = C * dx^(2τ)

    k_col_mat = [matern_kernel_2d([x[i], y[j]], [x[1], y[1]]; ν=ν, ℓ=ℓ, σf=σf)
                 for i in 1:Nx, j in 1:Ny]
    k_fft2 = bttb_fft_setup(k_col_mat, Nx, Ny)

    # ===== 直接法 (Cholesky) =====
    t_direct = @elapsed begin
        K     = [matern_kernel_2d(X_all[i], X_all[j]; ν=ν, ℓ=ℓ, σf=σf)
                 for i in 1:N, j in 1:N]
        A     = Dh_sp * K * Dh_sp' + σ_noise * I(N)
        α_dir = cholesky(Hermitian(A)) \ b_vec
        μ_dir = (Dh_sp * K)' * α_dir
    end
    mse_direct = mean((μ_dir .- u_true).^2)

    # ===== CG + dense DKD (competitor: Toeplitz性なし) =====
    if N <= dense_max_N
        t_cg_dense = @elapsed begin
            K_dense = [matern_kernel_2d(X_all[i], X_all[j]; ν=ν, ℓ=ℓ, σf=σf)
                       for i in 1:N, j in 1:N]
            DKD = Dh_sp * K_dense * Dh_sp' + σ_noise * I(N)
            α_dense, hist_dense = cg(Hermitian(DKD), b_vec;
                                     log=true, maxiter=50000, reltol=1e-10)
            μ_dense = (Dh_sp * K_dense)' * α_dense
        end
        mse_dense    = mean((μ_dense .- u_true).^2)
        n_iter_dense = hist_dense.iters
    else
        t_cg_dense   = NaN
        mse_dense    = NaN
        n_iter_dense = -1
    end

    # ===== CG + スパース + 2D FFT (提案手法) =====
    A_map = LinearMap(
        v -> begin
            w1 = Dh_sp' * v
            w2 = bttb_matvec(k_fft2, w1, Nx, Ny)
            w3 = Dh_sp * w2
            w3 .+ σ_noise .* v
        end,
        N; issymmetric=true, isposdef=true
    )
    t_cg = @elapsed begin
        α_cg, hist = cg(A_map, b_vec; log=true, maxiter=10N, reltol=1e-10)
        μ_cg = bttb_matvec(k_fft2, Dh_sp' * α_cg, Nx, Ny)
    end
    mse_cg = mean((μ_cg .- u_true).^2)
    n_iter = hist.iters

    push!(results, (
        N=N, Nx=Nx,
        t_direct=t_direct, t_cg_dense=t_cg_dense, t_cg=t_cg,
        mse_direct=mse_direct, mse_dense=mse_dense, mse_cg=mse_cg,
        n_iter=n_iter, n_iter_dense=n_iter_dense
    ))

    dense_str = isnan(t_cg_dense) ? "skipped" :
        @sprintf("%.3gs(iter=%d)", t_cg_dense, n_iter_dense)
    @printf("Nx=%3d (N=%5d): Cholesky=%.3gs  CG_dense=%s  CG_FFT=%.3gs(iter=%d)  MSE=%.3e\n",
        Nx, N, t_direct, dense_str, t_cg, n_iter, mse_cg)
end


In [ ]:
Ns           = [r.N           for r in results]
t_dir        = [r.t_direct    for r in results]
t_cg_dense_v = [r.t_cg_dense  for r in results]
t_cg_v       = [r.t_cg        for r in results]
mse_dir      = [r.mse_direct  for r in results]
mse_cg       = [r.mse_cg      for r in results]
n_iters      = [r.n_iter      for r in results]

mask_dense = .!isnan.(t_cg_dense_v)
Ns_dense   = Ns[mask_dense]
t_dense    = t_cg_dense_v[mask_dense]

p1 = plot(Ns, t_dir;
    marker=:circle, lw=2, label="Cholesky",
    xlabel="N = Nx²", ylabel="Time (s)",
    title="PFDM Helmholtz (k=$(k_wave)): Computation Time",
    yscale=:log10, xscale=:log10, legend=:topleft)
plot!(p1, Ns_dense, t_dense;
    marker=:diamond, lw=2, ls=:dashdot, color=:green,
    label="CG (dense DKD)")
plot!(p1, Ns, t_cg_v;
    marker=:square, lw=2, ls=:dash,
    label="CG (sparse + FFT)")
c3 = t_dir[end] / Ns[end]^3
c1 = t_cg_v[end] / (Ns[end] * log(Ns[end]))
plot!(p1, Ns, c3 .* Ns.^3;          lw=1, ls=:dot, color=:gray,   label="O(N³)")
plot!(p1, Ns, c1 .* Ns .* log.(Ns); lw=1, ls=:dot, color=:orange, label="O(N log N)")

p2 = plot(Ns, mse_dir; marker=:circle, lw=2, label="Cholesky",
    xlabel="N", ylabel="MSE", title="Posterior Mean MSE",
    yscale=:log10, xscale=:log10, legend=:topright)
plot!(p2, Ns, mse_cg; marker=:square, lw=2, ls=:dash, label="CG (FFT)")

p3 = plot(Ns, n_iters; marker=:diamond, lw=2, color=:purple,
    xlabel="N", ylabel="CG iterations", title="CG Iteration Count",
    xscale=:log10, legend=false)

p_all = plot(p1, p2, p3; layout=(1,3), size=(1200,400), margin=6Plots.mm)
display(p_all)
savefig(p_all, "pfdm_2d_helmholtz_bttb_cg.png")
println("Saved: pfdm_2d_helmholtz_bttb_cg.png")


## ハイパーパラメータ選択 (log周辺尤度グリッドサーチ)


In [ ]:
# log周辺尤度: log p(b | ℓ, σf, C)
# b = D_h u, モデル: b ~ N(0, D_h K D_h' + Σ)
function log_marginal_helmholtz(X_all, b, Dh_sp; ν, ℓ, σf, C, τ, dx, k_wave)
    N         = length(b)
    noise_var = C * dx^(2τ)
    K     = [matern_kernel_2d(X_all[i], X_all[j]; ν=ν, ℓ=ℓ, σf=σf)
             for i in 1:N, j in 1:N]
    A     = Dh_sp * K * Dh_sp' + noise_var * I(N) + 1e-10 * I(N)
    L     = cholesky(Hermitian(A)).L
    α     = L' \ (L \ b)
    logdet_A = 2sum(log.(diag(L)))
    return -0.5 * dot(b, α) - 0.5 * logdet_A - 0.5 * N * log(2π)
end

# グリッドサーチ設定 (Nx=10 で計算)
Nx_hp  = 10
Ny_hp  = Nx_hp
N_hp   = Nx_hp * Ny_hp
dx_hp  = 1.0 / (Nx_hp + 1)
τ_hp   = 2
ν_hp   = 1.5
k_hp   = 1.0   # Helmholtz 波数

x_hp   = range(dx_hp, step=dx_hp, length=Nx_hp)
y_hp   = range(dx_hp, step=dx_hp, length=Ny_hp)
X_hp   = [[xi, yj] for yj in y_hp for xi in x_hp]
Dh_hp  = build_Dh_2d_helmholtz(Nx_hp, Ny_hp, dx_hp, k_hp)
b_hp   = [f_func(xi, yj, k_hp) for yj in y_hp for xi in x_hp]

ℓ_grid   = 10 .^ range(-1, 1, length=10)
σf_grid  = 10 .^ range(-2, 1, length=10)
C_grid   = 10 .^ range(-4, 1, length=10)

best_score = -Inf
best_ℓ  = ℓ_grid[1]
best_σf = σf_grid[1]
best_C  = C_grid[1]

for ℓ in ℓ_grid, σf in σf_grid, C_val in C_grid
    score = log_marginal_helmholtz(X_hp, b_hp, Dh_hp;
                ν=ν_hp, ℓ=ℓ, σf=σf, C=C_val, τ=τ_hp, dx=dx_hp, k_wave=k_hp)
    if score > best_score
        best_score = score
        best_ℓ  = ℓ
        best_σf = σf
        best_C  = C_val
    end
end

@printf("best ℓ=%.4g  σf=%.4g  C=%.4g  loglik=%.4g\n",
    best_ℓ, best_σf, best_C, best_score)


In [ ]:
# ============================================================
# 2D可視化: best params で訓練(粗い) → テスト(密)で予測
# ============================================================
Nx_v    = Nx_hp
Nx_test = 50
Ny_v    = Nx_v
N_v     = Nx_v * Ny_v
N_test  = Nx_test^2
dx_v    = 1.0 / (Nx_v + 1)
dx_test = 1.0 / (Nx_test + 1)

# 訓練点
x_v  = range(dx_v, step=dx_v, length=Nx_v)
y_v  = range(dx_v, step=dx_v, length=Ny_v)
X_v  = [[xi, yj] for yj in y_v for xi in x_v]
b_v  = [f_func(xi, yj, k_wave) for yj in y_v for xi in x_v]
Dh_v = build_Dh_2d_helmholtz(Nx_v, Ny_v, dx_v, k_wave)
σ_v  = best_C * dx_v^(2τ)

# テスト点
x_t    = range(dx_test, step=dx_test, length=Nx_test)
y_t    = range(dx_test, step=dx_test, length=Nx_test)
X_t    = [[xi, yj] for yj in y_t for xi in x_t]
u_tr_t = [u_exact(xi, yj, k_wave) for yj in y_t for xi in x_t]

# カーネル行列
K_v    = [matern_kernel_2d(X_v[i], X_v[j]; ν=ν, ℓ=best_ℓ, σf=best_σf)
          for i in 1:N_v, j in 1:N_v]
K_star = [matern_kernel_2d(X_t[i], X_v[j]; ν=ν, ℓ=best_ℓ, σf=best_σf)
          for i in 1:N_test, j in 1:N_v]
K_ss_diag = [matern_kernel_2d(X_t[i], X_t[i]; ν=ν, ℓ=best_ℓ, σf=best_σf)
             for i in 1:N_test]

# posterior
A_v  = Dh_v * K_v * Dh_v' + σ_v * I(N_v)
A_ch = cholesky(Hermitian(A_v))
α_v  = A_ch \ b_v
μ_t  = K_star * (Dh_v' * α_v)

DhKs   = Dh_v * K_star'
LiDhKs = A_ch.L \ DhKs
var_t  = max.(K_ss_diag .- vec(sum(LiDhKs.^2, dims=1)), 0.0)
std_t  = sqrt.(var_t)
ci95_t = 1.96 .* std_t

# reshape
U_tr   = reshape(u_tr_t,              Nx_test, Nx_test)
U_mu   = reshape(μ_t,                 Nx_test, Nx_test)
U_err  = reshape(abs.(μ_t .- u_tr_t), Nx_test, Nx_test)
U_var  = reshape(var_t,               Nx_test, Nx_test)
U_ci95 = reshape(ci95_t,              Nx_test, Nx_test)

# カバレッジ確認
covered = abs.(μ_t .- u_tr_t) .<= ci95_t
coverage = mean(covered) * 100
@printf("95%% CI coverage: %.1f%%\n", coverage)
@printf("max |error|=%.3e  max 1.96σ=%.3e\n", maximum(U_err), maximum(U_ci95))

clim_sol = (0.0, maximum(U_tr))
clim_err = (0.0, max(maximum(U_err), maximum(U_ci95)))

hv1 = heatmap(collect(x_t), collect(y_t), U_tr';
    xlabel="x", ylabel="y", title="True solution",
    color=:viridis, clim=clim_sol)
hv2 = heatmap(collect(x_t), collect(y_t), U_mu';
    xlabel="x", ylabel="y", title="Posterior mean",
    color=:viridis, clim=clim_sol)
hv3 = heatmap(collect(x_t), collect(y_t), U_err';
    xlabel="x", ylabel="y", title="|True - Posterior mean|",
    color=:hot, clim=clim_err)
hv4 = heatmap(collect(x_t), collect(y_t), U_ci95';
    xlabel="x", ylabel="y", title="1.96 × Posterior std",
    color=:hot, clim=clim_err)
hv5 = heatmap(collect(x_t), collect(y_t), U_var';
    xlabel="x", ylabel="y", title="Posterior variance",
    color=:viridis)

# 3+2 レイアウト
fig2d = plot(hv1, hv2, hv3, hv4, hv5;
    layout=(2,3), size=(1200, 750), margin=5Plots.mm)
display(fig2d)
savefig(fig2d, "pfdm_2d_helmholtz_solution.png")
println("Saved: pfdm_2d_helmholtz_solution.png")
